# Experiment 10c - Block C inside the gated MLP (the dominant arm)

10b only put block C on the LGBM arm (28.5% of the ensemble), so C could barely move
the number. This puts C **into the gated MLP** - the 71.5% arm and your best pure-DL
model (14.59 CV / 14.75 LB) - which is where it actually matters.

Architecture, gates, and training are **identical to 7e** (so this is a clean test):
the only change is the expanded block-C dynamics added to the feature set. The MLP's
per-covariate gate auto-attaches each C feature to its parent covariate by name prefix
(`queue_pressure_forecast_vel1` -> `queue_pressure_forecast`), so no extra wiring.

**Run as a paired ablation:** no-C MLP vs +C MLP on the same folds and seeds, so the
delta is C's true effect on the MLP (untested before - C's LGBM gain may transfer
more, less, or not at all, since the MLP already has lag/lead/rolling features). Then a
submission: full-train MLP+C blended with the saved LGBM arm. **GPU required.**


---
## §0. Configuration (7e settings + block C + submission paths)

In [ ]:
from pathlib import Path
DATA_DIR=Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR=Path("/kaggle/working")
# Local: DATA_DIR=Path("data"); OUTPUT_DIR=Path("experiment-10")
LGBM_OOF_PATH=Path("/kaggle/input/exp9-oof/oof_predictions_9a_spike.csv")     # pred_mae (fold-aligned)
LGBM_VAL_PATH=Path("/kaggle/input/exp7-oof/val_components_7c.csv")            # pred_lgbm (validation)
# Local: experiment-9/oof_predictions_9a_spike.csv  /  experiment-7/val_components_7c.csv

TRAIN_PATH=DATA_DIR/"train.csv"; VAL_INPUT_PATH=DATA_DIR/"validation_input.csv"
SEED,N_FOLDS,EVAL_HORIZON=42,3,336
DOMAIN_FEAT_COLS=["workload_intensity","demand_forecast","upstream_quality_forecast","promotion_intensity",
  "shock_risk","maintenance_known","unit_reliability_forecast","queue_pressure_forecast",
  "network_pressure_forecast","event_load_forecast","service_irregularity_risk_forecast"]
TIME_COLS=["hour_sin","hour_cos","dow_sin","dow_cos","doy_sin","doy_cos","is_weekend"]
TOP_DOM=["queue_pressure_forecast","network_pressure_forecast","workload_intensity","shock_risk","event_load_forecast"]
TOP3=TOP_DOM[:3]; LAGS,LEADS,WINS=[1,24],[1,24],[6,24]; DYN_COVS=TOP_DOM
# 7e MLP config (verbatim)
MLP_HIDDEN=[256,256,128]; MLP_DROPOUT=0.2; MLP_EMBED_DIM=16; MLP_BATCH=1024
MLP_LR=1e-3; MLP_WEIGHT_DECAY=1e-4; MLP_EPOCHS=30; MLP_EARLY_STOP=5
GATE_L1_LAMBDA=0.01
CV_SEEDS=[42,101,202]; PROD_SEEDS=[42,101,202,303,404]
BEATS_GATED=14.59; ACCEPT_DELTA=0.05
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print("paired ablation: gated-MLP (no C) vs (+C); 7e architecture; GPU needed")


---
## §1. Setup + load

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEVICE)
def wape(y,p,eps=1e-8):
    y,p=np.asarray(y,float),np.asarray(p,float); return np.sum(np.abs(y-p))/(np.sum(np.abs(y))+eps)*100
train_df=pd.read_csv(TRAIN_PATH,parse_dates=["timestamp"]); val_input=pd.read_csv(VAL_INPUT_PATH,parse_dates=["timestamp"])
sid_to_code={s:i for i,s in enumerate(sorted(train_df.series_id.unique()))}; N_SERIES=len(sid_to_code)
print("train",train_df.shape,"val",val_input.shape,"series",N_SERIES)


---
## §2. Feature pipeline (baseline 7e + expanded block C) + gate maps

In [ ]:
def impute(df,cols):
    df=df.sort_values(["series_id","timestamp"]).copy()
    df[cols]=df.groupby("series_id",group_keys=False)[cols].transform(lambda x:x.ffill().bfill()); return df
def add_time(df):
    df=df.copy(); t=pd.to_datetime(df.timestamp)
    df["hour_sin"]=np.sin(2*np.pi*t.dt.hour/24); df["hour_cos"]=np.cos(2*np.pi*t.dt.hour/24)
    df["dow_sin"]=np.sin(2*np.pi*t.dt.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*t.dt.dayofweek/7)
    df["doy_sin"]=np.sin(2*np.pi*t.dt.dayofyear/365); df["doy_cos"]=np.cos(2*np.pi*t.dt.dayofyear/365)
    df["is_weekend"]=(t.dt.dayofweek>=5).astype(float); return df
def add_temporal(df):
    df=df.copy().sort_values(["series_id","timestamp"]); new=[]
    for c in TOP_DOM:
        g=df.groupby("series_id",sort=False)[c]
        for L in LAGS: k=f"{c}_lag{L}"; df[k]=g.shift(L); new.append(k)
        for L in LEADS: k=f"{c}_lead{L}"; df[k]=g.shift(-L); new.append(k)
    for c in TOP3:
        g=df.groupby("series_id",sort=False)[c]
        for W in WINS:
            km=f"{c}_roll{W}m"; df[km]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).mean()); new.append(km)
            ks=f"{c}_roll{W}s"; df[ks]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).std().fillna(0)); new.append(ks)
    return df,new
def add_dynamics_expanded(df):
    df=df.copy().sort_values(["series_id","timestamp"]); new=[]
    for c in DYN_COVS:
        g=df.groupby("series_id",sort=False)[c]
        df[f"{c}_vel1"]=g.diff(1); new.append(f"{c}_vel1")
        df[f"{c}_vel24"]=g.diff(24); new.append(f"{c}_vel24")
    for c in TOP3:
        g=df.groupby("series_id",sort=False)[c]
        df[f"{c}_acc1"]=g.diff(1).groupby(df.series_id,sort=False).diff(1); new.append(f"{c}_acc1")
        for span in (6,12,24):
            df[f"{c}_ewm{span}"]=g.transform(lambda x:x.shift(1).ewm(span=span,min_periods=1).mean()); new.append(f"{c}_ewm{span}")
    for c in ["queue_pressure_forecast","network_pressure_forecast"]:
        g=df.groupby("series_id",sort=False)[c]
        roll24=g.transform(lambda x:x.shift(1).rolling(24,min_periods=1).mean())
        expmed=g.transform(lambda x:x.shift(1).expanding(min_periods=1).median())
        df[f"{c}_regime"]=(roll24>expmed).astype(float); new.append(f"{c}_regime")
        p90=g.transform(lambda x:x.shift(1).expanding(min_periods=24).quantile(0.9))
        df[f"{c}_spike"]=(df[c]>p90).astype(float); new.append(f"{c}_spike")
    return df,new

def build_gate_idx_map(numeric_feats, cov_cols):
    out=[]
    for f in numeric_feats:
        if f in cov_cols: out.append(cov_cols.index(f))
        else:
            m=-1
            for ci,c in enumerate(cov_cols):
                if f.startswith(c+"_"): m=ci; break
            out.append(m)
    return out

base=add_time(impute(train_df,DOMAIN_FEAT_COLS)); base["sid_code"]=base.series_id.map(sid_to_code).astype("int32")
base,TEMP=add_temporal(base); base,DYN=add_dynamics_expanded(base)
NUMERIC_NOC=DOMAIN_FEAT_COLS+TIME_COLS+["series_mean"]+TEMP
NUMERIC_C  =NUMERIC_NOC+DYN
GATE_NOC=build_gate_idx_map(NUMERIC_NOC,DOMAIN_FEAT_COLS)
GATE_C  =build_gate_idx_map(NUMERIC_C,DOMAIN_FEAT_COLS)
print(f"no-C numeric {len(NUMERIC_NOC)} | +C numeric {len(NUMERIC_C)} (block C adds {len(DYN)})")
print(f"C features gated to a parent covariate: {sum(1 for i in GATE_C[len(NUMERIC_NOC):] if i>=0)}/{len(DYN)}")


---
## §3. MLP architecture + training (verbatim from 7e, gate_idx_map parametrised)

In [ ]:
class CovariateGate(nn.Module):
    def __init__(self,n_cov,gate_idx_map):
        super().__init__(); self.gates_logits=nn.Parameter(torch.zeros(n_cov))
        self.register_buffer("idx_map",torch.tensor(gate_idx_map,dtype=torch.long))
    def forward(self,x):
        g=torch.sigmoid(self.gates_logits); idx=self.idx_map.clamp(min=0); pf=g[idx]
        pf=torch.where(self.idx_map<0,torch.ones_like(pf),pf); return x*pf.unsqueeze(0)
    def values(self):
        with torch.no_grad(): return torch.sigmoid(self.gates_logits).cpu().numpy()
class GatedSeriesMLP(nn.Module):
    def __init__(self,n_numeric,n_series,embed_dim,hidden_dims,dropout,n_covariates,gate_idx_map):
        super().__init__(); self.gate=CovariateGate(n_covariates,gate_idx_map)
        self.series_emb=nn.Embedding(n_series,embed_dim); in_dim=n_numeric+embed_dim; layers=[]
        for h in hidden_dims: layers+=[nn.Linear(in_dim,h),nn.BatchNorm1d(h),nn.ReLU(inplace=True),nn.Dropout(dropout)]; in_dim=h
        layers.append(nn.Linear(in_dim,1)); self.net=nn.Sequential(*layers)
    def forward(self,x_num,sid):
        x=torch.cat([self.gate(x_num),self.series_emb(sid)],dim=1); return self.net(x).squeeze(-1)

def prepare_fold_tensors(tr_df,te_df,numeric_feats,sid_col="sid_code",target_col="target"):
    fs=RobustScaler().fit(tr_df[numeric_feats].astype(float))
    tr_X=np.nan_to_num(fs.transform(tr_df[numeric_feats].astype(float)),nan=0.0)
    te_X=np.nan_to_num(fs.transform(te_df[numeric_feats].astype(float)),nan=0.0)
    st=tr_df.groupby(sid_col)[target_col].agg(["mean","std"]).reset_index(); st["std"]=st["std"].fillna(1.0).clip(lower=1e-3)
    mu=st.set_index(sid_col)["mean"]; sd=st.set_index(sid_col)["std"]
    tr_mu=tr_df[sid_col].map(mu).to_numpy(); tr_sd=tr_df[sid_col].map(sd).to_numpy()
    te_mu=te_df[sid_col].map(mu).to_numpy(); te_sd=te_df[sid_col].map(sd).to_numpy()
    tr_y=(tr_df[target_col].to_numpy(float)-tr_mu)/tr_sd
    return dict(tr_X=torch.tensor(tr_X,dtype=torch.float32),te_X=torch.tensor(te_X,dtype=torch.float32),
        tr_sid=torch.tensor(tr_df[sid_col].to_numpy(np.int64)),te_sid=torch.tensor(te_df[sid_col].to_numpy(np.int64)),
        tr_y_norm=torch.tensor(tr_y,dtype=torch.float32),te_mu=te_mu,te_sd=te_sd)

def train_one_seed(ft,n_numeric,gate_idx_map,seed,l1_lambda=GATE_L1_LAMBDA):
    torch.manual_seed(seed); np.random.seed(seed)
    model=GatedSeriesMLP(n_numeric,N_SERIES,MLP_EMBED_DIM,MLP_HIDDEN,MLP_DROPOUT,len(DOMAIN_FEAT_COLS),gate_idx_map).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=MLP_LR,weight_decay=MLP_WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=MLP_EPOCHS); loss_fn=nn.L1Loss()
    n=ft["tr_X"].shape[0]; perm=torch.randperm(n,generator=torch.Generator().manual_seed(seed)); cut=int(0.9*n)
    tri,vi=perm[:cut],perm[cut:]
    ld=DataLoader(TensorDataset(ft["tr_X"][tri],ft["tr_sid"][tri],ft["tr_y_norm"][tri]),batch_size=MLP_BATCH,shuffle=True)
    vX=ft["tr_X"][vi].to(DEVICE); vS=ft["tr_sid"][vi].to(DEVICE); vy=ft["tr_y_norm"][vi].numpy()
    best,pat,bstate=float("inf"),0,None
    for ep in range(MLP_EPOCHS):
        model.train()
        for x,sid,y in ld:
            x,sid,y=x.to(DEVICE),sid.to(DEVICE),y.to(DEVICE); opt.zero_grad()
            loss=loss_fn(model(x,sid),y)+l1_lambda*torch.sum(torch.sigmoid(model.gate.gates_logits))
            loss.backward(); opt.step()
        sched.step(); model.eval()
        with torch.no_grad(): vp=model(vX,vS).cpu().numpy()
        vmae=float(np.mean(np.abs(vp-vy)))
        if vmae<best-1e-5: best=vmae; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; pat=0
        else:
            pat+=1
            if pat>=MLP_EARLY_STOP: break
    if bstate is not None: model.load_state_dict(bstate)
    model.eval()
    with torch.no_grad(): tp=model(ft["te_X"].to(DEVICE),ft["te_sid"].to(DEVICE)).cpu().numpy()
    return np.clip(tp*ft["te_sd"]+ft["te_mu"],0,None), model.gate.values()
print("MLP utils ready.")


---
## §4. Paired CV ablation - no-C MLP vs +C MLP

In [ ]:
def fold_off(k): return (N_FOLDS-k)*EVAL_HORIZON
def fold_split(end_off,length=EVAL_HORIZON):
    tr_l,te_l=[],[]
    for _,g in base.groupby("series_id",sort=False):
        g=g.sort_values("timestamp"); cut=len(g)-end_off; tr_l.append(g.iloc[:cut]); te_l.append(g.iloc[cut:cut+length])
    tr=pd.concat(tr_l,ignore_index=True); te=pd.concat(te_l,ignore_index=True)
    sm=tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm,on="series_id"),te.merge(sm,on="series_id")

def cv_mlp(numeric,gate,seeds=CV_SEEDS,label=""):
    pf=[]; oof=[]
    for k in range(N_FOLDS):
        tr,te=fold_split(fold_off(k)); ft=prepare_fold_tensors(tr,te,numeric)
        sp=[train_one_seed(ft,len(numeric),gate,s)[0] for s in seeds]
        avg=np.mean(sp,axis=0); pf.append(wape(te.target,avg))
        oof.append(te[["series_id","timestamp","target"]].assign(pred=avg,fold=k))
        print(f"   {label} fold {k}: WAPE {pf[-1]:.3f}")
    return pf, pd.concat(oof,ignore_index=True)

print("=== no-C MLP ===");  pf_noc, oof_noc = cv_mlp(NUMERIC_NOC, GATE_NOC, label="noC")
print("=== +C MLP ===");    pf_c,   oof_c   = cv_mlp(NUMERIC_C,   GATE_C,   label="+C")
m_noc,m_c=np.mean(pf_noc),np.mean(pf_c)
print(f"\nno-C MLP : folds {[round(x,3) for x in pf_noc]}  mean {m_noc:.3f}  (7e ref 14.59)")
print(f"+C  MLP  : folds {[round(x,3) for x in pf_c]}  mean {m_c:.3f}")
print(f"Δ (C helps MLP?): {m_noc-m_c:+.3f}  ({'ACCEPT' if m_noc-m_c>ACCEPT_DELTA else '~noise'})")


---
## §5. Blend re-opt (MLP+C OOF + saved LGBM OOF) + residual r

In [ ]:
lg=pd.read_csv(LGBM_OOF_PATH,parse_dates=["timestamp"])[["series_id","timestamp","fold","pred_mae"]]
J=oof_c.rename(columns={"pred":"pred_mlp_c"}).merge(lg,on=["series_id","timestamp","fold"],how="inner")
assert len(J)==len(oof_c)
r=float(np.corrcoef(J.target-J.pred_mae, J.target-J.pred_mlp_c)[0,1]); print(f"r(MLP+C, LGBM) = {r:.4f}")
pa=J.pred_mlp_c.to_numpy(); pb=J.pred_mae.to_numpy()
def cvw(w):
    J["_b"]=w*pa+(1-w)*pb; v=np.mean([wape(g.target,g["_b"]) for _,g in J.groupby("fold")]); J.drop(columns="_b",inplace=True); return float(v)
grid=np.round(np.arange(0,1.0001,0.005),3); w_star,best=min(((w,cvw(w)) for w in grid),key=lambda t:t[1])
print(f"MLP+C solo {cvw(1.0):.3f} | LGBM solo {cvw(0.0):.3f} | best blend {best:.3f} at w_mlp={w_star:.3f}")
print(f"(shipped ensemble: CV 14.47 / LB 14.68)")


---
## §6. Submission - full-train MLP+C, predict validation, blend with saved LGBM

In [ ]:
val_stub=val_input.copy(); val_stub["target"]=np.nan
comb=pd.concat([train_df,val_stub],ignore_index=True)
comb["is_val"]=np.r_[np.zeros(len(train_df),bool),np.ones(len(val_stub),bool)]
comb=add_time(impute(comb,DOMAIN_FEAT_COLS)); comb["sid_code"]=comb.series_id.map(sid_to_code).astype("int32")
comb,_=add_temporal(comb); comb,_=add_dynamics_expanded(comb)
sm=comb[~comb.is_val].groupby("series_id")["target"].mean().rename("series_mean").reset_index(); comb=comb.merge(sm,on="series_id")
tr_full=comb[~comb.is_val]; va=comb[comb.is_val].sort_values(["series_id","timestamp"])
ft=prepare_fold_tensors(tr_full,va,NUMERIC_C)
vp=[train_one_seed(ft,len(NUMERIC_C),GATE_C,s)[0] for s in PROD_SEEDS]
pred_mlp_c=np.mean(vp,axis=0)
sub=va[["series_id","timestamp"]].copy(); sub["pred_mlp_c"]=pred_mlp_c
lgv=pd.read_csv(LGBM_VAL_PATH,parse_dates=["timestamp"]); lcol="pred_lgbm" if "pred_lgbm" in lgv.columns else "prediction"
sub=sub.merge(lgv[["series_id","timestamp",lcol]].rename(columns={lcol:"pred_lgbm"}),on=["series_id","timestamp"],how="left")
assert sub.pred_lgbm.notna().all()
sub["prediction"]=w_star*sub.pred_mlp_c+(1-w_star)*sub.pred_lgbm
print("submission rows",len(sub),"NaN:",sub.prediction.isna().any()); print(sub.head())


---
## §7. Save + verdict

In [ ]:
sub[["series_id","timestamp","prediction"]].sort_values(["series_id","timestamp"]).to_csv(OUTPUT_DIR/"submission_10c_mlpC_ensemble.csv",index=False)
oof_c.to_csv(OUTPUT_DIR/"oof_predictions_10c_mlp_c.csv",index=False)
print("submission ->",OUTPUT_DIR/"submission_10c_mlpC_ensemble.csv")
print("\n=== Verdict (block C inside the MLP) ===")
print(f"  no-C MLP {m_noc:.3f} | +C MLP {m_c:.3f} | Δ {m_noc-m_c:+.3f}")
print(f"  ensemble (MLP+C + LGBM) best CV {best:.3f} at w_mlp={w_star:.3f}  (shipped 14.47/LB14.68)")
if m_noc-m_c>ACCEPT_DELTA:
    print("  C HELPS the MLP -> this is a real improvement; submit and compare LB vs 14.68/14.75.")
elif abs(m_noc-m_c)<=ACCEPT_DELTA:
    print("  C ~ noise inside the MLP too (the lag/lead/rolling features already encode most of it).")
    print("  Submit if best CV <= 14.47; else block C is confirmed exhausted across BOTH arms.")
else:
    print("  C HURTS the MLP (added noise/overfit). Keep the no-C MLP; block C is closed.")
